# Independent shear bias comparison: GalSim vs BATSim

Runs each simulator's pipeline independently — auto-detect sources, reverse-map to
COSMOS galaxies, measure shear — without requiring that the two simulators detect the
same objects. The difference in multiplicative bias *m* between GalSim and BATSim
therefore captures both measurement differences and selection effects naturally.

Contrast with `new_shear_accuracy.ipynb`, which restricts analysis to the intersection
of galaxies detected by both simulators.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import galsim
import batsim
import anacal
import numpy as np
import matplotlib.pyplot as plt
import pickle

from tqdm import tqdm
from importlib import reload
reload(batsim)

In [ ]:
def measure_shear(gal_array, psf_array, npix, pixel_scale, noise_variance, noise_array, detection):
    """
    Measure shear using the FPFS estimator via anacal.

    Parameters
    ----------
    gal_array : np.ndarray
        2D galaxy image array.
    psf_array : np.ndarray
        2D PSF image array.
    npix : int
        Individual galaxy stamp size in pixels.
    pixel_scale : float
        Pixel scale in arcsec/pixel.
    noise_variance : float
        Noise variance; a minimum floor of 0.23 is applied internally.
    noise_array : np.ndarray or None
        2D noise image, or None.
    detection : np.ndarray or None
        Structured array with fields ('y', 'x') for pre-specified positions,
        or None to run auto-detection.

    Returns
    -------
    np.ndarray
        Structured catalog with FPFS per-galaxy measurements including
        'fpfs_w', 'fpfs_e1', 'fpfs_de1_dg1', 'fpfs_dw_dg1'.
    """
    fpfs_config = anacal.fpfs.FpfsConfig(
        sigma_shapelets=0.52,
        sigma_shapelets2=0.45,
        npix=npix,
    )
    return anacal.fpfs.process_image(
        fpfs_config=fpfs_config,
        mag_zero=30.0,
        gal_array=gal_array,
        psf_array=psf_array,
        pixel_scale=pixel_scale,
        noise_variance=max(noise_variance, 0.23),
        noise_array=noise_array,
        detection=detection,
    )


def jackknife_over_tiles(E_tiles, R_tiles, lens_shear):
    """
    Estimate multiplicative bias and uncertainty using tile-based jackknife.

    Parameters
    ----------
    E_tiles : array_like
        Per-tile sums of weighted ellipticity (fpfs_w * fpfs_e1).
    R_tiles : array_like
        Per-tile sums of shear response.
    lens_shear : float
        True input shear g1; used to compute m = g1_hat / g1 - 1.

    Returns
    -------
    dict
        Keys: 'g1', 'g1_err', 'm', 'm_err', 'E_tiles', 'R_tiles', 'g1_loo', 'm_loo'.
    """
    E_tiles = np.asarray(E_tiles)
    R_tiles = np.asarray(R_tiles)
    E_tot = E_tiles.sum()
    R_tot = R_tiles.sum()
    g1 = E_tot / R_tot
    m  = g1 / lens_shear - 1
    n  = len(E_tiles)
    g1_loo = (E_tot - E_tiles) / (R_tot - R_tiles)
    m_loo  = g1_loo / lens_shear - 1
    g1_err = np.sqrt((n - 1) / n * np.sum((g1_loo - g1_loo.mean()) ** 2))
    m_err  = np.sqrt((n - 1) / n * np.sum((m_loo  - m_loo.mean())  ** 2))
    return {'g1': g1, 'g1_err': g1_err, 'm': m, 'm_err': m_err,
            'E_tiles': E_tiles, 'R_tiles': R_tiles,
            'g1_loo': g1_loo, 'm_loo': m_loo}

In [ ]:
np.random.seed(14)

cosmos = galsim.COSMOSCatalog()
print(f'Catalog size: {len(cosmos)}')

# Must match the parameters used to generate the saved stamps
scale          = 0.2
nn             = 96
lens_shear     = 0.02
n_rot          = 4
n_gal_stamp    = 1600
tiles_per_side = 40
tile_size      = int(nn * np.sqrt(n_rot))                    # 192
stamp_size     = int(nn * np.sqrt(n_rot * n_gal_stamp))      # 7680

n_stamps   = int(np.ceil(len(cosmos) / n_gal_stamp))
all_inds   = np.arange(len(cosmos))
stamp_inds = np.array_split(all_inds, n_stamps)
print(f'Number of stamps: {n_stamps}')

psf       = galsim.Moffat(beta=3.5, fwhm=0.7)
psf_array = psf.drawImage(nx=nn, ny=nn, scale=scale).array

In [ ]:
galsim_stamps = []
batsim_stamps = []

for i in tqdm(range(n_stamps)):
    gs_file = os.path.join('COSMOS_lensed', f'galsim_stamp_{i}_{stamp_size}x{stamp_size}.fits')
    bt_file = os.path.join('COSMOS_lensed', f'batsim_stamp_{i}_{stamp_size}x{stamp_size}.fits')
    galsim_stamps.append(galsim.fits.read(file_name=gs_file))
    batsim_stamps.append(galsim.fits.read(file_name=bt_file))

print(f'Loaded {len(galsim_stamps)} GalSim and {len(batsim_stamps)} BATSim stamps')

In [ ]:
all_gal_inds = np.concatenate(stamp_inds)
all_records  = cosmos.getParametricRecord(index=all_gal_inds)

mag_all      = all_records['mag_auto']
hlr_all      = all_records['hlr'][:, 0]        # half-light radius [arcsec]
sersic_n_all = all_records['sersicfit'][:, 2]  # Sersic index
is_bulgefit  = all_records['use_bulgefit'].astype(bool)

stamp_offsets = np.concatenate([[0], np.cumsum([len(s) for s in stamp_inds])])

print(f'Total galaxies  : {len(all_gal_inds)}')
print(f'mag range       : [{mag_all.min():.1f}, {mag_all.max():.1f}]')
print(f'hlr range       : [{hlr_all.min():.3f}, {hlr_all.max():.3f}] arcsec')
print(f'sersic_n range  : [{sersic_n_all.min():.1f}, {sersic_n_all.max():.1f}]')
print(f'bulge+disk frac : {is_bulgefit.mean():.2%}')

In [ ]:
POS_Y = 'y'
POS_X = 'x'


def detection_to_galaxy_k(cat, nn, n_rot, tiles_per_side=40):
    """
    Reverse-map auto-detected source positions to local galaxy indices.

    Each galaxy occupies a tile of size (nn * sqrt(n_rot)) pixels. Integer
    division of the detected position by the tile size gives the tile row
    and column, from which the local galaxy index k is recovered.

    Parameters
    ----------
    cat : np.ndarray
        Structured catalog with position fields POS_Y and POS_X.
    nn : int
        Individual galaxy stamp size in pixels (e.g. 96).
    n_rot : int
        Number of shape-noise cancellation rotations (e.g. 4).
    tiles_per_side : int, optional
        Number of galaxy tiles per side of the large stamp. Default is 40.

    Returns
    -------
    np.ndarray of int
        Local galaxy index k (0-indexed within the stamp) for each catalog entry.
        Multiple entries can share the same k if FPFS finds more than one peak
        in a tile; these are kept as-is to reflect true pipeline behaviour.
    """
    ts = int(nn * np.sqrt(n_rot))
    y  = cat[POS_Y].astype(int)
    x  = cat[POS_X].astype(int)
    return (y // ts) * tiles_per_side + (x // ts)

In [ ]:
CACHE_DIR = 'COSMOS_lensed'

gs_catalogs = []
bt_catalogs = []
gs_k_maps   = []   # per-stamp local galaxy index for each GalSim detection
bt_k_maps   = []   # per-stamp local galaxy index for each BATSim detection

for i in tqdm(range(n_stamps), total=n_stamps):
    cache_file = os.path.join(CACHE_DIR, f'fpfs_independent_v2_{i}.pkl')
    n_gal = len(stamp_inds[i])

    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            gs_cat, k_gs, bt_cat, k_bt = pickle.load(f)
    else:
        gs_cat = measure_shear(
            gal_array=galsim_stamps[i].array, psf_array=psf_array,
            npix=nn, pixel_scale=scale, noise_variance=0.23,
            noise_array=None, detection=None,
        )
        bt_cat = measure_shear(
            gal_array=batsim_stamps[i].array, psf_array=psf_array,
            npix=nn, pixel_scale=scale, noise_variance=0.23,
            noise_array=None, detection=None,
        )

        k_gs = detection_to_galaxy_k(gs_cat, nn, n_rot)
        k_bt = detection_to_galaxy_k(bt_cat, nn, n_rot)

        # Drop detections that fall outside valid galaxy tiles
        gs_cat, k_gs = gs_cat[k_gs < n_gal], k_gs[k_gs < n_gal]
        bt_cat, k_bt = bt_cat[k_bt < n_gal], k_bt[k_bt < n_gal]

        # No deduplication: all FPFS detections are kept (typically ~n_rot per galaxy)
        # so that shape noise cancellation works and full pipeline behaviour is preserved.

        with open(cache_file, 'wb') as f:
            pickle.dump((gs_cat, k_gs, bt_cat, k_bt), f)

    gs_catalogs.append(gs_cat)
    bt_catalogs.append(bt_cat)
    gs_k_maps.append(k_gs)
    bt_k_maps.append(k_bt)

n_gs_det  = sum(len(k) for k in gs_k_maps)
n_bt_det  = sum(len(k) for k in bt_k_maps)
n_gs_gals = sum(len(np.unique(k)) for k in gs_k_maps)
n_bt_gals = sum(len(np.unique(k)) for k in bt_k_maps)
n_total   = len(all_gal_inds)
print(f'GalSim : {n_gs_det:7d} detections  ({n_gs_gals:6d} unique galaxies, {n_gs_gals/n_total:.1%})')
print(f'BATSim : {n_bt_det:7d} detections  ({n_bt_gals:6d} unique galaxies, {n_bt_gals/n_total:.1%})')

In [ ]:
# Global boolean detection flags — used later for per-bin detection rates
gs_detected = np.zeros(len(all_gal_inds), dtype=bool)
bt_detected = np.zeros(len(all_gal_inds), dtype=bool)
for i in range(n_stamps):
    if len(gs_k_maps[i]) > 0:
        gs_detected[stamp_offsets[i] + gs_k_maps[i]] = True
    if len(bt_k_maps[i]) > 0:
        bt_detected[stamp_offsets[i] + bt_k_maps[i]] = True


def collect_props_for_mask(det_mask_global):
    """
    Collect galaxy properties for galaxies flagged in a global boolean mask.

    Parameters
    ----------
    det_mask_global : np.ndarray of bool
        Boolean array of length len(all_gal_inds). True selects a galaxy.

    Returns
    -------
    dict
        Keys: 'mag', 'hlr', 'sersic_n', 'is_bulgefit'.
    """
    return {
        'mag':         mag_all[det_mask_global],
        'hlr':         hlr_all[det_mask_global],
        'sersic_n':    sersic_n_all[det_mask_global],
        'is_bulgefit': is_bulgefit[det_mask_global],
    }


props_gs = collect_props_for_mask(gs_detected)
props_bt = collect_props_for_mask(bt_detected)

# Property distributions of what each simulator detects
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Detected galaxy property distributions: GalSim vs BATSim', fontsize=12)

for ax, key, bins, xlabel in [
    (axes[0], 'mag',      np.linspace(18, 26,  30), 'mag_auto'),
    (axes[1], 'hlr',      np.linspace(0,  2.0, 30), 'r_e  [arcsec]'),
    (axes[2], 'sersic_n', np.linspace(0,  8,   30), 'Sersic n'),
]:
    for label, p, color, ls in [
        ('GalSim', props_gs, 'C0', '-'),
        ('BATSim', props_bt, 'C1', '--'),
    ]:
        counts, edges = np.histogram(p[key], bins=bins)
        centres = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centres, counts / counts.sum(), color=color, ls=ls, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('fraction')
    ax.legend()

plt.tight_layout()
plt.savefig('independent_detection_properties.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def compute_bias_for_mask(galaxy_mask):
    """
    Compute jackknife multiplicative bias independently for GalSim and BATSim.

    Each simulator is evaluated on its own detected sample within the mask.
    No matching between simulators is performed: GalSim's k-map is used for
    GalSim's catalog and BATSim's k-map for BATSim's catalog.

    Parameters
    ----------
    galaxy_mask : np.ndarray of bool
        Boolean array of length len(all_gal_inds). True includes a galaxy.

    Returns
    -------
    res_gs : dict or None
        Jackknife result for GalSim (see jackknife_over_tiles). None if the
        total shear response for GalSim is zero within the mask.
    res_bt : dict or None
        Jackknife result for BATSim. None under the same conditions.
    """
    E_gs, R_gs, E_bt, R_bt = [], [], [], []
    for i in range(n_stamps):
        for e_list, R_list, cat, k_map in [
            (E_gs, R_gs, gs_catalogs[i], gs_k_maps[i]),
            (E_bt, R_bt, bt_catalogs[i], bt_k_maps[i]),
        ]:
            if len(k_map) == 0:
                e_list.append(0.0)
                R_list.append(0.0)
                continue
            det_mask = galaxy_mask[stamp_offsets[i] + k_map]
            e = (cat['fpfs_w'] * cat['fpfs_e1'])[det_mask]
            R = (cat['fpfs_dw_dg1'] * cat['fpfs_e1']
                 + cat['fpfs_w'] * cat['fpfs_de1_dg1'])[det_mask]
            e_list.append(e.sum())
            R_list.append(R.sum())

    res_gs = jackknife_over_tiles(E_gs, R_gs, lens_shear) if np.sum(R_gs) != 0 else None
    res_bt = jackknife_over_tiles(E_bt, R_bt, lens_shear) if np.sum(R_bt) != 0 else None
    return res_gs, res_bt


def run_binned_analysis(prop, bin_edges, label, base_mask=None):
    """
    Measure GalSim and BATSim multiplicative bias in bins of a galaxy property.

    Each simulator is evaluated independently on its own detected sample within
    each bin. Detection rates (fraction of bin galaxies detected) are also
    computed for each simulator separately.

    Parameters
    ----------
    prop : np.ndarray
        Property values for all galaxies, in all_gal_inds order.
    bin_edges : np.ndarray
        Bin edges; bins are half-open [edge[i], edge[i+1]).
    label : str
        Label shown in the tqdm progress bar.
    base_mask : np.ndarray of bool or None, optional
        Additional mask AND-ed with each bin mask. Default is None.

    Returns
    -------
    dict
        Keys: 'centers', 'bin_edges', 'm_gs', 'm_gs_err', 'm_bt', 'm_bt_err',
              'dm', 'dm_err', 'n_in_bin', 'det_rate_gs', 'det_rate_bt'.
    """
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    n_bins  = len(centers)
    out = dict(
        centers=centers, bin_edges=bin_edges,
        m_gs=np.full(n_bins, np.nan), m_gs_err=np.full(n_bins, np.nan),
        m_bt=np.full(n_bins, np.nan), m_bt_err=np.full(n_bins, np.nan),
        n_in_bin=np.zeros(n_bins, dtype=int),
        det_rate_gs=np.full(n_bins, np.nan),
        det_rate_bt=np.full(n_bins, np.nan),
    )
    for b in tqdm(range(n_bins), desc=label):
        mask = (prop >= bin_edges[b]) & (prop < bin_edges[b + 1])
        if base_mask is not None:
            mask &= base_mask
        out['n_in_bin'][b] = mask.sum()
        if out['n_in_bin'][b] < 10:
            continue
        out['det_rate_gs'][b] = (mask & gs_detected).sum() / out['n_in_bin'][b]
        out['det_rate_bt'][b] = (mask & bt_detected).sum() / out['n_in_bin'][b]
        res_gs, res_bt = compute_bias_for_mask(mask)
        if res_gs is None or res_bt is None:
            continue
        out['m_gs'][b], out['m_gs_err'][b] = res_gs['m'], res_gs['m_err']
        out['m_bt'][b], out['m_bt_err'][b] = res_bt['m'], res_bt['m_err']
    out['dm']     = out['m_bt'] - out['m_gs']
    out['dm_err'] = np.sqrt(out['m_gs_err'] ** 2 + out['m_bt_err'] ** 2)
    return out

In [ ]:
# Full-sample bias: sum over every detection in each simulator's catalog,
# no reverse-mapping or masking. Matches the approach in the first part of
# test_shear_accuracy_real.ipynb and serves as the reference for the binned plots.
E_gs, R_gs, E_bt, R_bt = [], [], [], []
for gs_cat, bt_cat in zip(gs_catalogs, bt_catalogs):
    for e_list, R_list, cat in [(E_gs, R_gs, gs_cat), (E_bt, R_bt, bt_cat)]:
        e = cat['fpfs_w'] * cat['fpfs_e1']
        R = (cat['fpfs_dw_dg1'] * cat['fpfs_e1']
             + cat['fpfs_w'] * cat['fpfs_de1_dg1'])
        e_list.append(e.sum())
        R_list.append(R.sum())

res_full_gs = jackknife_over_tiles(E_gs, R_gs, lens_shear)
res_full_bt = jackknife_over_tiles(E_bt, R_bt, lens_shear)

print(f'True g1 = {lens_shear}')
print(f'GalSim  g1 = {res_full_gs["g1"]:.6f} ± {res_full_gs["g1_err"]:.6f}  |  m = {res_full_gs["m"]:.4e} ± {res_full_gs["m_err"]:.4e}')
print(f'BATSim  g1 = {res_full_bt["g1"]:.6f} ± {res_full_bt["g1_err"]:.6f}  |  m = {res_full_bt["m"]:.4e} ± {res_full_bt["m_err"]:.4e}')

In [ ]:
res_mag = run_binned_analysis(
    mag_all, np.array([18, 20, 21, 22, 23, 24, 25, 25.3]), 'magnitude'
)

res_n = run_binned_analysis(
    sersic_n_all, np.array([0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]),
    'Sersic n', base_mask=~is_bulgefit
)

res_hlr = run_binned_analysis(
    hlr_all, np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.2, 2.0]),
    'half-light radius'
)

In [ ]:
def get_valid(res, ref_err, max_err_factor=5):
    """
    Build a boolean mask selecting bins with reliable bias estimates.

    Parameters
    ----------
    res : dict
        Output from run_binned_analysis.
    ref_err : float
        Full-sample jackknife error used as the reference scale.
    max_err_factor : float, optional
        Maximum allowed ratio of bin error to ref_err. Default is 5.

    Returns
    -------
    np.ndarray of bool
        True for bins that are not NaN and within the error threshold.
    """
    return (
        ~np.isnan(res['m_gs'])
        & (res['m_gs_err'] < max_err_factor * ref_err)
        & (res['m_bt_err'] < max_err_factor * ref_err)
    )


def plot_independent_summary(results_list, titles, xlabels, log_x_flags,
                              res_full_gs, res_full_bt, save_path=None):
    """
    Plot independent multiplicative bias comparison for multiple properties.

    Produces a 3-row x N-column figure. Row 0 shows m_gs and m_bt with
    full-sample reference bands. Row 1 shows Δm = m_bt - m_gs. Row 2 shows
    the detection rate (fraction of bin galaxies detected) per simulator.

    Parameters
    ----------
    results_list : list of dict
        Output dicts from run_binned_analysis, one per property.
    titles : list of str
        Column titles.
    xlabels : list of str
        x-axis labels.
    log_x_flags : list of bool
        Whether to use a log x-axis for each column.
    res_full_gs : dict
        Full-sample GalSim jackknife result, used for reference bands.
    res_full_bt : dict
        Full-sample BATSim jackknife result, used for reference bands.
    save_path : str or None, optional
        File path to save the figure. Default is None.

    Returns
    -------
    None
    """
    n_cols = len(results_list)
    fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 10), sharey='row',
                             gridspec_kw={'height_ratios': [2, 1.5, 1]})
    fig.suptitle('Independent shear bias vs galaxy properties', fontsize=13)

    ref_lo = min((res_full_gs['m'] - 3 * res_full_gs['m_err']) * 1e3,
                 (res_full_bt['m'] - 3 * res_full_bt['m_err']) * 1e3)
    ref_hi = max((res_full_gs['m'] + 3 * res_full_gs['m_err']) * 1e3,
                 (res_full_bt['m'] + 3 * res_full_bt['m_err']) * 1e3)
    pad = abs(ref_hi - ref_lo)

    for col, (res, title, xlabel, log_x) in enumerate(
        zip(results_list, titles, xlabels, log_x_flags)
    ):
        valid  = get_valid(res, res_full_gs['m_err'])
        x      = res['centers'][valid]

        # Row 0: bias values
        ax = axes[0, col]
        ax.axhspan((res_full_gs['m'] - res_full_gs['m_err']) * 1e3,
                   (res_full_gs['m'] + res_full_gs['m_err']) * 1e3,
                   alpha=0.1, color='C0', label='GalSim full \u00b11\u03c3')
        ax.axhspan((res_full_bt['m'] - res_full_bt['m_err']) * 1e3,
                   (res_full_bt['m'] + res_full_bt['m_err']) * 1e3,
                   alpha=0.1, color='C1', label='BATSim full \u00b11\u03c3')
        ax.errorbar(x, res['m_gs'][valid] * 1e3, yerr=res['m_gs_err'][valid] * 1e3,
                    fmt='o-', capsize=3, color='C0', label='GalSim')
        ax.errorbar(x, res['m_bt'][valid] * 1e3, yerr=res['m_bt_err'][valid] * 1e3,
                    fmt='s-', capsize=3, color='C1', label='BATSim')
        ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.5)
        ax.set_title(title)
        ax.set_ylim(ref_lo - pad, ref_hi + pad)
        if col == 0:
            ax.set_ylabel('m  [\u00d710\u207b\u00b3]')
        ax.legend(fontsize=8)

        # Row 1: delta-m
        ax2 = axes[1, col]
        ax2.errorbar(x, res['dm'][valid] * 1e3, yerr=res['dm_err'][valid] * 1e3,
                     fmt='D-', color='C2', capsize=3)
        ax2.axhline(0, color='k', ls='--', lw=0.8, alpha=0.5)
        if col == 0:
            ax2.set_ylabel('\u0394m = m_bt \u2212 m_gs  [\u00d710\u207b\u00b3]')

        # Row 2: detection rate
        ax3 = axes[2, col]
        ax3.plot(x, res['det_rate_gs'][valid] * 100, 'o-', color='C0', label='GalSim')
        ax3.plot(x, res['det_rate_bt'][valid] * 100, 's-', color='C1', label='BATSim')
        ax3.set_xlabel(xlabel)
        ax3.set_ylim(0, 105)
        if col == 0:
            ax3.set_ylabel('Detection rate (%)')
        ax3.legend(fontsize=8)

        if log_x:
            for ax_ in axes[:, col]:
                ax_.set_xscale('log')

    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_independent_summary(
    results_list=[res_mag,      res_n,                          res_hlr],
    titles       =['Magnitude',  'Sersic n (single-component)',  'Half-light radius'],
    xlabels      =['mag_auto',   'n',                            'r_e  [arcsec]'],
    log_x_flags  =[False,        False,                          True],
    res_full_gs  =res_full_gs,
    res_full_bt  =res_full_bt,
    save_path    ='independent_bias_summary.png',
)